# 带收益收集的车辆路径问题 (PCVRP)

**类别：** 路径

来源： [https://www.hexaly.com/templates/prize-collecting-vehicle-routing-problem-pcvrp](https://www.hexaly.com/templates/prize-collecting-vehicle-routing-problem-pcvrp)


## 问题

**在 Prize-Collecting Vehicle Routing Problem (PCVRP) 中**，一组具有相同容量的配送车辆必须为对某种共同商品有已知需求的客户提供服务。车辆从一个共同的配送中心出发并最终返回该配送中心，每辆车服务的总需求量不得超过其容量。每位客户最多只能被一辆车访问。无需访问所有客户，但需要满足一个最低的需求量。此外，每位被访问的客户会带来一定的奖励。有三个优化目标：最小化车队规模、最大化所收集的奖励、最小化总行驶距离。

	

### 学到的建模原则

- 添加 [list decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模每辆卡车的客户访问顺序
- 定义 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算行驶距离和所收集的奖励
- 添加 [multiple objectives](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html)


## 数据

我们提供的 Prize-Collecting Vehicle Routing Problem (PCVRP) 实例来自 [Long et al. Set A instances](https://github.com/longjianyuGH/PCVRP.git)。他们为 [Augerat et al. Set A instances](http://neo.lcc.uma.es/vrp/vrp-instances/capacitated-vrp-instances/) 添加了奖励值以及必须满足的最低需求量。数据文件的格式如下：

- 第一行：车辆数量、车辆容量、必须满足的最低需求量
- 第二行：配送中心的 ID 和坐标
- 接下来几行，对于每位客户：该客户的 ID、坐标、需求和奖励


## 程序

Prize-Collecting Vehicle Routing Problem (PCVRP) 的 Hexaly 模型使用 list decision variables。对于每辆卡车，我们定义一个 list 变量，表示该卡车访问的客户顺序。通过在所有 list 上使用 **disjoint** 约束，我们确保每位客户最多被一辆卡车服务。

如果一辆卡车至少访问一位客户，则该卡车在车队中被使用。借助 ‘count’ 算子（返回 list 中的元素个数），我们可以检查每辆卡车是否被使用，然后计算车队中使用的卡车总数。

每辆卡车交付的总需求量通过 [**lambda function**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算，对所有访问的客户应用 ‘sum’ 算子。注意此求和中项的数量以及 list 的大小在搜索过程中是变化的。然后我们可以约束该数量不超过卡车容量。类似地，我们计算每辆卡车在其路线上收集的总奖励。

从一位客户到下一位客户的行驶距离同样通过在二维距离矩阵上使用 ‘at’ 算子来访问。我们使用另一个 [**lambda function**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 对每辆卡车在其路线上的距离进行求和，从而计算其总行驶距离。

三个目标按字典序进行优化。我们首先最小化使用的卡车数量，然后最大化所收集的总奖励，最后最小化所有卡车的总行驶距离。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def main(instance_file, str_time_limit, output_file):
    #
    # Read instance data
    #
    nb_customers, nb_trucks, truck_capacity, dist_matrix_data, dist_depot_data, \
        demands_data, demands_to_satisfy, prizes_data = read_input_pcvrp(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Sequence of customers visited by each truck
        customers_sequences = [model.list(nb_customers) for _ in range(nb_trucks)]

        # A customer might be visited by only one truck
        model.constraint(model.disjoint(customers_sequences))

        # Create Hexaly arrays to be able to access them with an "at" operator
        demands = model.array(demands_data)
        prizes = model.array(prizes_data)
        dist_matrix = model.array(dist_matrix_data)
        dist_depot = model.array(dist_depot_data)

        # A truck is used if it visits at least one customer
        trucks_used = [(model.count(customers_sequences[k]) > 0) for k in range(nb_trucks)]

        dist_routes = [None] * nb_trucks
        route_prizes = [None] * nb_trucks
        route_quantities = [None] * nb_trucks

        for k in range(nb_trucks):
            sequence = customers_sequences[k]
            c = model.count(sequence)

            # The quantity needed in each route must not exceed the truck capacity
            demand_lambda = model.lambda_function(lambda j: demands[j])
            route_quantities[k] = model.sum(sequence, demand_lambda)
            model.constraint(route_quantities[k] <= truck_capacity)

            # Distance traveled by each truck
            dist_lambda = model.lambda_function(lambda i:
                                                model.at(dist_matrix,
                                                         sequence[i - 1],
                                                         sequence[i]))
            dist_routes[k] = model.sum(model.range(1, c), dist_lambda) \
                + model.iif(c > 0,
                            dist_depot[sequence[0]] + dist_depot[sequence[c - 1]],
                            0)
            
            # Route prize of truck k
            prize_lambda = model.lambda_function(lambda j: prizes[j])
            route_prizes[k] = model.sum(sequence, prize_lambda)

        # Total nb demands satisfied
        total_quantity = model.sum(route_quantities)
    
        # Minimal number of demands to satisfy
        model.constraint(total_quantity >= demands_to_satisfy)

        # Total nb trucks used
        nb_trucks_used = model.sum(trucks_used)

        # Total distance traveled
        total_distance = model.sum(dist_routes)

        # Total prize
        total_prize = model.sum(route_prizes)

        # Objective: minimize the number of trucks used, then maximize the total prize and minimize the distance traveled
        model.minimize(nb_trucks_used)
        model.maximize(total_prize)
        model.minimize(total_distance)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(str_time_limit)

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - total prize, number of trucks used and total distance
        #  - for each truck the customers visited (omitting the start/end at the depot)
        #  - number of unvisited customers, demands satisfied
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d %d %d\n" % (total_prize.value, nb_trucks_used.value, total_distance.value))
                nb_unvisited_customers = nb_customers
                for k in range(nb_trucks):
                    if trucks_used[k].value != 1:
                        continue
                    # Values in sequence are in 0...nbCustomers. +1 is to put it back in 1...nbCustomers+1
                    # as in the data files (0 being the depot)
                    for customer in customers_sequences[k].value:
                        f.write("%d " % (customer + 1))
                        nb_unvisited_customers -= 1
                    f.write("\n")
                f.write("%d %d\n" % (nb_unvisited_customers, total_quantity.value))


# The input files follow the "longjianyu" format
def read_input_pcvrp(filename):
    file_it = iter(read_elem(filename))

    nb_trucks = int(next(file_it))
    truck_capacity = int(next(file_it))
    demands_to_satisfy = int(next(file_it))

    n = 0
    customers_x = []
    customers_y = []
    depot_x = 0
    depot_y = 0
    demands = []
    prizes = []
    
    it = next(file_it, None)
    while (it != None):
        node_id = int(it)
        if node_id != n:
            print("Unexpected index")
            sys.exit(1)

        if n == 0:
            depot_x = int(next(file_it))
            depot_y = int(next(file_it))
        else:
            customers_x.append(int(next(file_it)))
            customers_y.append(int(next(file_it)))
            demands.append(int(next(file_it)))
            prizes.append(int(next(file_it)))
        it = next(file_it, None)
        n += 1

    distance_matrix = compute_distance_matrix(customers_x, customers_y)
    distance_depots = compute_distance_depots(depot_x, depot_y, customers_x, customers_y)

    nb_customers = n - 1

    return nb_customers, nb_trucks, truck_capacity, distance_matrix, distance_depots, \
        demands, demands_to_satisfy, prizes


# Compute the distance matrix
def compute_distance_matrix(customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_matrix = [[None for _ in range(nb_customers)] for _ in range(nb_customers)]
    for i in range(nb_customers):
        distance_matrix[i][i] = 0
        for j in range(nb_customers):
            dist = compute_dist(customers_x[i], customers_x[j], customers_y[i], customers_y[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    return distance_matrix


# Compute the distances to depot
def compute_distance_depots(depot_x, depot_y, customers_x, customers_y):
    nb_customers = len(customers_x)
    distance_depots = [None] * nb_customers
    for i in range(nb_customers):
        dist = compute_dist(depot_x, customers_x[i], depot_y, customers_y[i])
        distance_depots[i] = dist
    return distance_depots


def compute_dist(xi, xj, yi, yj):
    exact_dist = math.sqrt(math.pow(xi - xj, 2) + math.pow(yi - yj, 2))
    return int(math.floor(exact_dist + 0.5))


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python pcvrp.py input_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "20"

    main(instance_file, str_time_limit, output_file)
